# DWR Processing Toolkit — Quickstart Demo

This notebook walks through loading a single IMD DWR Level-2 NetCDF file,
applying quality control, and visualising the corrected fields.

**Data source:** MOSDAC — https://mosdac.gov.in/  
Place a sample `.nc` file in `INPUT_DATA/` before running.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import wradlib as wrl

from dwr_processing.io.netcdf_reader import (
    read_imd_netcdf,
    detect_polarisation_type,
    extract_scan_metadata,
    get_scan_mode,
)
from dwr_processing.clutter.filters import gabella_filter, polarimetric_filter
from dwr_processing.velocity.dealias import dealias_velocity

%matplotlib inline

## 1. Load a radar file

In [ ]:
# Update this path to point to your IMD DWR NetCDF file
FILEPATH = "../INPUT_DATA/YOUR_FILE_HERE.nc"

raw = read_imd_netcdf(FILEPATH)
pol_type = detect_polarisation_type(raw)
meta = extract_scan_metadata(raw)
mode = get_scan_mode(meta["num_elev"])

print(f"Polarisation type : {pol_type}")
print(f"Scan mode         : {mode}")
print(f"Site location     : lat={meta['site_lat']:.3f}°, lon={meta['site_lon']:.3f}°, alt={meta['site_alt']:.0f} m")
print(f"Num elevations    : {meta['num_elev']}")
print(f"Num range gates   : {meta['num_bins']}")
print(f"Num azimuths      : {meta['num_azim']}")
print(f"Nyquist velocity  : {meta['nyquist']} m/s")

## 2. Apply clutter filtering to reflectivity

In [ ]:
dbz_raw = raw["variables"]["Z"]["data"]

if pol_type == "single_pol":
    dbz_clean = gabella_filter(dbz_raw)
else:
    rhohv = raw["variables"]["RHOHV"]["data"]
    phidp = raw["variables"]["PHIDP"]["data"]
    vel   = raw["variables"]["V"]["data"]
    zdr   = raw["variables"]["ZDR"]["data"]
    dbz_clean = polarimetric_filter(dbz_raw, rhohv, phidp, vel, zdr)

print("Clutter filtering complete.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, title in zip(
    axes,
    [dbz_raw, dbz_clean],
    ["Raw Reflectivity (dBZ)", "Clutter-Filtered Reflectivity (dBZ)"],
):
    im = ax.pcolormesh(data, cmap="pyart_NWSRef", vmin=-10, vmax=70)
    ax.set_title(title)
    ax.set_xlabel("Range gate")
    ax.set_ylabel("Azimuth")
    plt.colorbar(im, ax=ax, label="dBZ")

plt.tight_layout()
plt.savefig("reflectivity_qc.png", dpi=150)
plt.show()

## 3. Velocity dealiasing (short-range scans)

In [ ]:
if mode == "short_range" and "V" in raw["variables"]:
    vel_raw = raw["variables"]["V"]["data"].astype(float)
    fill_val = raw["variables"]["V"].get("_FillValue")
    v_n = float(meta["nyquist"])
    range_res = float(raw["variables"]["gateSize"]["data"])

    # Mask clutter positions in velocity
    vel_raw[np.isnan(dbz_clean)] = np.nan
    if fill_val is not None:
        vel_raw[vel_raw == fill_val] = np.nan

    vel_dealiased = dealias_velocity(
        velocity=vel_raw,
        v_n=v_n,
        range_resolution=range_res,
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, data, title in zip(
        axes,
        [vel_raw, vel_dealiased],
        ["Raw Velocity (m/s)", "Dealiased Velocity (m/s)"],
    ):
        im = ax.pcolormesh(data, cmap="RdBu_r", vmin=-v_n, vmax=v_n)
        ax.set_title(title)
        ax.set_xlabel("Range gate")
        ax.set_ylabel("Azimuth")
        plt.colorbar(im, ax=ax, label="m/s")

    plt.tight_layout()
    plt.savefig("velocity_qc.png", dpi=150)
    plt.show()
else:
    print("Velocity dealiasing skipped (long-range scan or V not available).")

## 4. Next steps

- Run the full batch pipeline: `python scripts/quality_control_imd_multiproc.py`
- See `docs/algorithm_notes.md` for details on the clutter and dealiasing algorithms.
- Download more IMD DWR data from [MOSDAC](https://mosdac.gov.in/).
